In [1]:
import torch
from torch.autograd import Variable

In [2]:
x = torch.ones(2, 2, requires_grad=True)
print(x)

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)


In [3]:
x.register_hook(lambda grad:grad*2)

In [4]:
y = x + 2
z = y * y * 3
# z = torch.sum(z)
# nn = torch.rand(2, 2)
nn = torch.ones(2, 2)
print(nn)

tensor([[1., 1.],
        [1., 1.]])


In [5]:
z.backward(gradient=nn, retain_graph=True)
torch.autograd.backward(z, grad_tensors=nn, retain_graph=True)

print(torch.autograd.grad(z, [x, y, z], grad_outputs=nn))

(tensor([[36., 36.],
        [36., 36.]]), tensor([[18., 18.],
        [18., 18.]]), tensor([[1., 1.],
        [1., 1.]]))


In [6]:
print(x.grad)

tensor([[72., 72.],
        [72., 72.]])


In [7]:
print(y.grad)

None


/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_7757/1324337507.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print(y.grad)


In [8]:
print(x.grad_fn)

None


In [9]:
print(y.grad_fn)

In [10]:
print(z.grad_fn)

In [11]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
# Original tensor: tensor([1., 2.], requires_grad=True)
print("Original tensor:", x)

Original tensor: tensor([1., 2.], requires_grad=True)


In [12]:
# tensor operation
y = x + 2  # add
z = y * y * 3  # multiplication
out = z.mean()  # mean

In [13]:
# compute gradients
out.backward()
print("Gradient of x with respect to the output:", x.grad)

Gradient of x with respect to the output: tensor([ 9., 12.])


In [15]:
'''
In some scenarios, such as when validating the model or calculating some intermediate results that do not require updating parameters, 
preventing gradient tracking can reduce memory consumption and improve efficiency. Using .detach() or torch.no_grad() are effective means of achieving this.
'''
# # Using the .detach() method: Returns a new tensor with the same value as the original tensor, but does not track 
# gradients.new_tensor = x.detach()

# # Use torch.no_grad() context manager.
# with torch.no_grad():
#     # Operations performed in this area will not track gradients
#     intermediate_result = some_operation(original_tensor)

'\nIn some scenarios, such as when validating the model or calculating some intermediate results that do not require updating parameters, \npreventing gradient tracking can reduce memory consumption and improve efficiency. Using .detach() or torch.no_grad() are effective means of achieving this.\n'

In [16]:
# Prevent gradient tracing
# In some scenarios, such as when validating the model or calculating some intermediate results that do not require updating parameters, preventing gradient tracking can reduce memory consumption and improve efficiency. Using .detach() or torch.no_grad() are effective means of achieving this.

# method 1 - Using the .detach() method: Returns a new tensor with the same value as the original tensor, but does not track gradients.
# new_tensor = original_tensor.detach()

# # method2 - Use torch.no_grad() context manager.
# with torch.no_grad():
#     # Operations performed in this area will not track gradients
#     intermediate_result = some_operation(original_tensor)


In [17]:
# A context manager 
# controls gradient calculations torch.autograd.set_grad_enabled(True|False) is another powerful tool for globally controlling whether gradient calculations are performed in specific parts of the code. Compared to .detach() and torch.no_grad(), it provides more flexibility because it allows gradient tracking to be dynamically turned on or off in different parts of the code, which is useful for complex model debugging, performance optimization, or mixed precision training Especially useful in other scenarios.
# default senario, track gradients
# print(f"track current gradient {torch.is_grad_enabled()}")  # output: True

In [18]:
# set_grad_enabled(False)
with torch.autograd.set_grad_enabled(False):
    x = torch.tensor([1.0, 2.0], requires_grad=True)
    y = x * 2
    print(f"is current gradient enabled: {torch.is_grad_enabled()}")  # Output: False
    print(f"y status: {y.requires_grad}")  #OUtput: False

is current gradient enabled: False
y status: False


In [19]:
# Leave context, recover tracking gradient status
print(f"Without context，track gradient status: {torch.is_grad_enabled()}")  # 输出: True

Without context，track gradient status: True


In [20]:
from torch import nn, optim

In [24]:
import torch
from torch import nn, optim

# ==============================
# 1. Create a toy dataset
# ==============================
# 10 samples, 2 features (x1, x2)
torch.manual_seed(42)
inputs = torch.randn(10, 2)

# True relationship: y = 2*x1 + 3*x2 + 1 + noise
true_w = torch.tensor([[2.0], [3.0]])
true_b = 1.0
labels = inputs @ true_w + true_b + 0.1 * torch.randn(10, 1)

# ==============================
# 2. Define model, loss, optimizer
# ==============================
model = nn.Linear(2, 1)  # 2 input features → 1 output
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# ==============================
# 3. Training loop
# ==============================
for epoch in range(100):
    # Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, labels)

    # Backward pass
    optimizer.zero_grad()  # reset gradients
    loss.backward()        # compute new gradients
    optimizer.step()       # update parameters

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

# ==============================
# 4. Inspect learned parameters
# ==============================
print("\nLearned weights and bias:")
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

print("\nTrue weights and bias:")
print(f"weights: {true_w.squeeze().tolist()}, bias: {true_b}")


Epoch [10/100], Loss: 0.1843
Epoch [20/100], Loss: 0.0148
Epoch [30/100], Loss: 0.0023
Epoch [40/100], Loss: 0.0010
Epoch [50/100], Loss: 0.0009
Epoch [60/100], Loss: 0.0009
Epoch [70/100], Loss: 0.0009
Epoch [80/100], Loss: 0.0009
Epoch [90/100], Loss: 0.0009
Epoch [100/100], Loss: 0.0009

Learned weights and bias:
weight: tensor([[1.9350, 3.0269]])
bias: tensor([1.0297])

True weights and bias:
weights: [2.0, 3.0], bias: 1.0
